# Práctica 1: Transformación de formatos a coordenadas tridimensionales. Grafos moleculares y representaciones químicas

> **Bloque 1 — Generación de estructuras y espacio conformacional** · Manual de QC · UNAM

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eduardo-Gabriel-Guzman-Lopez/computational-chemistry-book/blob/main/notebooks/p01.ipynb)

In [ ]:
# ── Configuración ─────────────────────────────────────────────
# Google Colab: descomenta la siguiente línea
# !pip install rdkit-pypi xtb-python umap-learn -q

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams["figure.dpi"] = 120
print("Entorno listo ✓  —  Manual QC UNAM")


Transformación de formatos a coordenadas tridimensionales.
Grafos moleculares y representaciones químicas

### 📋 Información de la práctica

| Parámetro | Valor |
|:--|:--|
| **Bloque temático** | 1: Generación de estructuras
 y espacio conformacional |
| **Número de práctica** |  |
| **Nivel de dificultad** | {Básico |
| **Tiempo estimado** | 1 h (semilla) + 2 h (bosque y análisis) |
| **Modalidad** | Individual |
| **Pipeline central** | SMILES $\to$ grafo molecular $\to$ geometría 3D (ETKDG)
 $\to$ pre-opt FF (MMFF94) $\to$ opt semiempírica (GFN2-xTB)
 $\to$ dataset estructural |
| **Hardware mínimo** | 2 GB RAM, procesador de doble núcleo $\geq$ 1.5 GHz
 (laptop de 2012 o posterior) |
| **Modo sin conexión** | Parcial — internet requerida solo para
 instalación inicial; la práctica completa
 funciona offline una vez instalado el entorno |
| **Acceso en nube** | {https://colab.research.google.com/drive/p01_cafeina |

## Introducción química

Antes de realizar cualquier cálculo cuántico, una molécula debe existir
como un objeto computacional con coordenadas tridimensionales explícitas.
Este paso, aparentemente trivial, encierra decisiones metodológicas con
consecuencias directas sobre la calidad de todos los cálculos posteriores:
una geometría inicial mal construida puede converger a un mínimo incorrecto,
producir frecuencias imaginarias espurias o generar resultados numéricos
que no corresponden a ninguna especie química real.

La representación más común en quimioinformática es el *SMILES*
(*Simplified Molecular Input Line Entry System*), una cadena de
texto que codifica la conectividad molecular mediante reglas gramaticales
precisas. Un SMILES como 'CC(=O)Nc1ccccc1' describe sin ambigüedad
la acetanilida; sin embargo, no contiene información sobre ángulos de
enlace, longitudes de enlace ni disposición espacial. Pasar de esa cadena
lineal a un conjunto de coordenadas cartesianas $(x_i, y_i, z_i)$ es el
problema central de esta práctica.

Las herramientas modernas —principalmente RDKit y OpenBabel— implementan
algoritmos de incrustación tridimensional que combinan reglas geométricas
empíricas, campos de fuerza y, en el caso del algoritmo ETKDG
(*Experimental-Torsion distance geometry with basic Knowledge*),
distribuciones de ángulos diedros extraídas de la Cambridge Structural
Database {cite}`rdkit,etkdg`. El resultado es una geometría de partida
razonable en pocos milisegundos, sin importar el tamaño de la molécula.

Una vez disponible la geometría inicial, la práctica introduce el primer
pipeline del manual: pre-optimización con un campo de fuerza (MMFF94)
seguida de optimización semiempírica con GFN2-xTB {cite}`grimme_xtb`.
Este protocolo de dos pasos es suficiente para la mayoría de las moléculas
orgánicas pequeñas y medianas, y constituye el punto de partida estándar
para los cálculos de estructura electrónica del Bloque~2.

En términos del modelo Semilla–Bosque: la **semilla** es la
construcción y optimización de una molécula sencilla que el estudiante
ejecuta por sí mismo. El **bosque** es un dataset de 50 moléculas
orgánicas con diversidad estructural —aromáticos, heteroaromáticos,
sistemas flexibles y casos difíciles— para los cuales el mismo pipeline
ya fue ejecutado. El análisis del bosque permite identificar qué
características estructurales hacen que la incrustación 3D sea más o
menos confiable, y cómo la energía de pre-optimización se correlaciona
con descriptores topológicos básicos.


 SMILES (texto),
 Grafo molecular (RDKit),
 Geometría 3D — ETKDG,
 Pre-opt FF (MMFF94),
 Opt semiempírica (GFN2-xTB),
 Dataset estructural

## Marco teórico

### Conceptos clave

**1. Representaciones moleculares y su jerarquía informacional.**
Una molécula puede codificarse en distintos niveles de detalle. La fórmula
molecular (0D) sólo indica composición. El SMILES o InChI (1D) codifican
la conectividad y el orden de enlace. El grafo 2D añade estereoquímica
plana. Las coordenadas cartesianas 3D son las únicas que permiten calcular
propiedades que dependen de la geometría (energías, frecuencias, densidades
electrónicas). Cada nivel es una proyección: al bajar de 3D a 1D se pierde
información irreversiblemente.

**2. Geometría de distancias (DG) e incrustación 3D.**
El algoritmo ETKDG convierte un grafo molecular en coordenadas cartesianas
resolviendo un problema de optimización sobre matrices de distancias
interatómicas. Los límites de cada distancia se obtienen de tres fuentes:
reglas de enlace covalente, restricciones de no-penetración van der Waals,
y distribuciones estadísticas de ángulos diedros extraídas de estructuras
cristalinas. El resultado no es único: distintas semillas aleatorias
producen geometrías de partida diferentes, todas igualmente válidas como
punto de inicio.

**3. Campos de fuerza (FF) y su rol como pre-optimizadores.**
Un campo de fuerza es una función de energía potencial empírica que
describe la energía como suma de contribuciones de enlace, ángulo,
torsión e interacciones no covalentes. MMFF94 está parametrizado para
moléculas orgánicas pequeñas y es adecuado para corregir geometrías
iniciales ETKDG. No es apropiado para describir reacciones, rupturas
de enlace o sistemas con metales de transición.

**4. Métodos semiempíricos ajustados (GFN2-xTB).**
GFN2-xTB es un método de enlace fuerte autoconsistente (*tight-binding*)
parametrizado para reproducir geometrías y energías de conformación de
moléculas orgánicas e inorgánicas con precisión comparable a DFT/def2-SVP,
pero con un costo computacional 3–4 órdenes de magnitud menor {cite}`grimme_xtb`.
La energía total de GFN2-xTB contiene contribuciones electrónicas,
de dispersión (D4) y de repulsión nuclear. No es apropiado para excitaciones
electrónicas ni para propiedades que requieren descripción explícita de
la función de onda.

**5. RMSD como métrica de similitud geométrica.**
El RMSD (*Root Mean Square Deviation*) entre dos geometrías se define
como:

$$
    \mathrm{RMSD} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}
    \left|\mathbf{r}_i^{(A)} - \mathbf{r}_i^{(B)}\right|^2}
$$

donde $N$ es el número de átomos y los vectores $\mathbf{r}_i$ son las
coordenadas de cada átomo tras la superposición óptima de las dos
estructuras. Un RMSD $< 0.5$~ entre la geometría calculada y el
cristal indica reproducción aceptable.

### Preguntas previas

1. **(Conceptual)** El SMILES de la cafeína no contiene  coordenadas. Sin embargo, tú ya sabes que la cafeína tiene  un anillo purínico. ¿Cómo "sabe" el algoritmo ETKDG que  ese anillo debe ser plano? ¿Qué pasaría si la molécula tuviera  un anillo que *no* aparece frecuentemente en la CSD?
2. **(Predictivo)** El campo de fuerza MMFF94 fue  desarrollado principalmente para moléculas orgánicas con  C, H, N, O, S. La cafeína tiene cuatro átomos de nitrógeno  en distintos entornos (amida, imina, amina aromática).  ¿Esperas que MMFF94 describa bien todos esos entornos?  ¿Cuál crees que será el enlace con mayor error antes de  la corrección xTB?
3. **(Crítico)** El bosque de esta práctica incluye  diez "casos difíciles": moléculas muy tensionadas como  el cubano o espiro-compuestos. ¿Por qué crees que la  incrustación automática podría fallar en esos sistemas?  ¿Qué alternativa propondrías para construir su geometría?

### Recursos de consulta rápida

- **RDKit Getting Started (en línea, inglés):**  [https://www.rdkit.org/docs/GettingStartedInPython.html](https://www.rdkit.org/docs/GettingStartedInPython.html) —  cubre los módulos 'Chem', 'AllChem' y  'rdMolDescriptors' que se usan en esta práctica.
- **Tutorial xTB (en línea, inglés):**  [https://xtb-docs.readthedocs.io/en/latest/setup.html](https://xtb-docs.readthedocs.io/en/latest/setup.html) —  instalación, opciones de línea de comandos y ejemplos de  optimización.
- **Química computacional sin matemáticas, Lewars (2024):**  Capítulo~2 (representaciones moleculares) y Capítulo~3  (campos de fuerza). Disponible en la biblioteca del curso.
- **Dataset público QM9:**  [https://doi.org/10.6084/m9.figshare.978904](https://doi.org/10.6084/m9.figshare.978904) —  133~885 moléculas orgánicas pequeñas con geometrías DFT.  Útil como contexto para entender el tamaño del espacio químico  explorado en el bosque de esta práctica.

## Objetivos de aprendizaje

### Conceptuales

- Distinguir entre representaciones moleculares 0D, 1D, 2D y 3D,  e identificar qué información contiene y qué omite cada una.
- Explicar el principio del algoritmo ETKDG y por qué produce  geometrías diferentes con distintas semillas aleatorias.
- Justificar el protocolo FF $\to$ xTB como compromiso entre  velocidad y calidad para el punto de partida de cálculos DFT.

### Procedimentales

- Instalar y usar RDKit y xtb desde un script de Python y desde  la línea de comandos.
- Convertir entre formatos moleculares ('.smi',  '.xyz', '.sdf') usando RDKit y OpenBabel.
- Ejecutar una optimización GFN2-xTB, verificar la convergencia  en el log y extraer la geometría resultante.
- Visualizar y comparar estructuras 3D en Avogadro2 o py3Dmol.

### Investigación y datos

- Construir un dataset de 50 moléculas con las variables:  SMILES, número de átomos pesados, enlaces rotativos,  energía GFN2-xTB, tiempo de optimización y éxito/fallo de  la incrustación.
- Identificar qué clases estructurales presentan mayor tasa de  fallo en la incrustación 3D automática.
- Correlacionar la energía de la geometría optimizada con  descriptores topológicos básicos del bosque.

## Recursos y prerrequisitos

### Conocimientos previos

- **(Necesario)** Nomenclatura IUPAC de orgánicos básicos.
- **(Necesario)** Enlace covalente, hibridación y geometría  molecular (TRPEV/VSEPR).
- **(Necesario)** Terminal de Linux/macOS: navegar  directorios, ejecutar scripts, redirigir salida.
- **(Necesario)** Python: variables, listas, bucles  'for', importar módulos.
- **(Recomendable)** Haber leído el Capítulo~1 del manual  (Estructura y flujos de trabajo).

```{admonition} 📝 Nota metodológica
:class: note

**Primera práctica del manual.** No se asumen conocimientos previos
de química computacional. Si nunca has usado la terminal, completa
primero el tutorial 'docs/terminal\_basica.md' del repositorio
del curso (tiempo estimado: 30 min).
```

### Software

- **Python** $\geq$ 3.10 con 'rdkit' $\geq$ 2023.03,  'numpy', 'pandas', 'matplotlib',  'seaborn'.  Instalar entorno completo: verbatim conda env create -f environment_p01.yml conda activate qc-manual verbatim
- **xtb** $\geq$ 6.6 — open-source, gratuito (LGPL-3).  Grupo Grimme, Universidad de Bonn. verbatim conda install -c conda-forge xtb verbatim
- **OpenBabel** $\geq$ 3.1 — open-source, gratuito (GPL-2).  Para conversiones de formato. verbatim conda install -c conda-forge openbabel verbatim
- **Avogadro2** o **py3Dmol** — visualización 3D,  ambos gratuitos y multiplataforma.

In [ ]:
%%bash

conda env create -f environment_p01.yml
conda activate qc-manual

**xtb** $\geq$ 6.6 — open-source, gratuito (LGPL-3).
 Grupo Grimme, Universidad de Bonn.

In [ ]:
%%bash

conda install -c conda-forge xtb

**OpenBabel** $\geq$ 3.1 — open-source, gratuito (GPL-2).
 Para conversiones de formato.

In [ ]:
%%bash

conda install -c conda-forge openbabel

**Avogadro2** o **py3Dmol** — visualización 3D,
 ambos gratuitos y multiplataforma.
itemize

```{admonition} 📝 Nota metodológica
:class: note

**Esta práctica no requiere Gaussian, ORCA ni ningún software
comercial.** Todo el pipeline (ETKDG + MMFF94 + GFN2-xTB) se ejecuta
con herramientas completamente open-source. Las prácticas del Bloque~2
en adelante introducirán ORCA (gratuito para uso académico) como
herramienta principal de estructura electrónica.
```

```{admonition} ⚠️ Advertencia
:class: warning

**¿Sin acceso a software local?**
El notebook de Google Colab incluye toda la instalación automatizada y
el dataset del bosque. No requiere configuración previa:
[https://colab.research.google.com/drive/p01_cafeina](https://colab.research.google.com/drive/p01_cafeina)
```

### Archivos proporcionados

- 'p01\_semilla.py' — script base para la semilla.
- 'p01\_bosque\_smiles.csv' — 50 SMILES con nombre  y clase estructural.
- 'p01\_bosque\_resultados.csv' — dataset pre-calculado.
- 'p01\_analisis.py' — script base de análisis.
- 'environment\_p01.yml' — entorno conda reproducible.
- 'p01\_colab.ipynb' — notebook para Google Colab.

### Recursos computacionales y accesibilidad

### 📋 Información de la práctica

| Parámetro | Valor |
|:--|:--|
| **Semilla — CPU** | $< 2$ min en 1 núcleo (laptop estándar) |
| **Bosque — CPU** | $\sim$ 15 min en 1 núcleo (pre-calculado) |
| **RAM mínima absoluta** | 2 GB (laptop de 2012 o posterior) |
| **Espacio en disco** | $< 50$ MB |
| **Modo sin conexión** | Sí, una vez instalado el entorno conda |
| **Acceso en nube** | {https://colab.research.google.com/drive/p01_cafeina |

```{admonition} 📝 Nota metodológica
:class: note

**Accesibilidad:** Todos los scripts de análisis generan figuras
con etiquetas textuales en ejes y títulos descriptivos, compatibles
con lectores de pantalla. Las tablas de resultados se exportan como CSV
con nombres de columna en español. Para adaptaciones específicas
(discapacidad visual, daltonismo, etc.), consulta
'docs/accesibilidad.md' en el repositorio del curso.
```

## Experimento semilla

### Sistema modelo

La semilla de esta práctica es la **cafeína**
('Cn1cnc2c1c(=O)n(c(=O)n2C)C', CAS 58-08-2), elegida por cuatro
razones simultáneas: (i) tiene 24 átomos pesados —pequeña pero no
trivial—; (ii) posee un núcleo heterocíclico aromático y grupos carbonilo
que ponen a prueba el campo de fuerza; (iii) su estructura cristalina
está bien determinada por difracción de rayos X, lo que permite
validación cuantitativa {cite}`caffeine_crystal`; y (iv) es una molécula
que todos los estudiantes conocen, lo que facilita la conexión con la
intuición química previa.

### Pregunta química concreta

### Nivel de teoría

### 📋 Información de la práctica

| Parámetro | Valor |
|:--|:--|
| **Método de incrustación** | ETKDG v3 (RDKit 2023) |
| **Campo de fuerza** | MMFF94 (pre-optimización) |
| **Método semiempírico** | GFN2-xTB (optimización final) |
| **Solvente** | Vacío (fase gas) |
| **Convergencia** | Gradiente RMS $< 5 10^{-4 |
| **Justificación** | GFN2-xTB reproduce longitudes de enlace de orgánicos
 con error medio $< 0.01$~ respecto a DFT/def2-SVP, con costo
 3–4 órdenes de magnitud menor {cite}`grimme_xtb` |

### Hipótesis inicial

Basándome en las preguntas previas, espero que el anillo purínico sea
plano (o casi plano) en la geometría GFN2-xTB, porque los sistemas
aromáticos están sobrerrepresentados en la CSD y el algoritmo ETKDG
los reconoce bien. El mayor error debería aparecer en los grupos
metilo ($-$CH$_3$), cuya posición rotacional tiene energía plana y
el algoritmo puede colocar en una orientación sub-óptima. Esta hipótesis
será verificable comparando el RMSD de los C del anillo frente al
RMSD de los carbonos de los metilos respecto al cristal.

## Protocolo computacional

### 1. Construcción del grafo molecular

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolDescriptors

smiles = 'Cn1cnc2c1c(=O)n(c(=O)n2C)C'
nombre = 'cafeina'

mol = Chem.MolFromSmiles(smiles)
if mol is None:
    raise ValueError(f'SMILES no válido: {smiles}')

print(f'Átomos pesados : {mol.GetNumHeavyAtoms()}')
print(f'Fórmula        : {rdMolDescriptors.CalcMolFormula(mol)}')
print(f'Peso molecular : '
      f'{rdMolDescriptors.CalcExactMolWt(mol):.4f} Da')
print(f'Enlaces rot.   : '
      f'{rdMolDescriptors.CalcNumRotatableBonds(mol)}')

# Guardar imagen 2D del grafo
Draw.MolToImageFile(mol, f'{nombre}_2D.png', size=(400, 300))

### 2. Incrustación 3D con ETKDG

In [ ]:
mol_h  = Chem.AddHs(mol)          # hidrógenos explícitos
params = AllChem.ETKDGv3()
params.randomSeed = 42             # reproducibilidad

ok = AllChem.EmbedMolecule(mol_h, params)
if ok == -1:
    raise RuntimeError(
        'Falló la incrustación 3D. '
        'Prueba con maxAttempts=200 o ETKDGv3.')
print('Incrustación exitosa.')

```{admonition} ⚠️ Advertencia
:class: warning

**Error frecuente:** 'EmbedMolecule' devuelve $-1$ cuando
la molécula tiene quiralidad o tensión de anillo inusuales. Si ocurre,
añade 'params.maxAttempts = 200' antes de llamar a
'EmbedMolecule'.
```

### 3. Pre-optimización con MMFF94

In [ ]:
props = AllChem.MMFFGetMoleculeProperties(mol_h)
ff    = AllChem.MMFFGetMoleculeForceField(mol_h, props)

if ff is None:          # fallback si MMFF94 no está parametrizado
    print('MMFF94 no disponible, usando UFF.')
    ff = AllChem.UFFGetMoleculeForceField(mol_h)

conv = ff.Minimize(maxIts=2000)
print(f'Energía MMFF94: {ff.CalcEnergy():.4f} kcal/mol')
print(f'Convergencia (0=OK): {conv}')

from rdkit.Chem import rdmolfiles
rdmolfiles.MolToXYZFile(mol_h, f'{nombre}_FF.xyz')

### 4. Optimización semiempírica con GFN2-xTB

In [ ]:
%%bash

# Desde la terminal (bash):
xtb cafeina_FF.xyz --opt tight --chrg 0 --uhf 0 \
    --gfn 2 > cafeina_xtb.out 2>&1

# Verificar convergencia:
grep "GEOMETRY OPTIMIZATION CONVERGED" cafeina_xtb.out

# Archivos generados:
#   xtbopt.xyz   — geometría final (renombrar a cafeina_GFN2.xyz)
#   xtbopt.log   — trayectoria completa
#   charges      — cargas atómicas de Mulliken

Parámetros críticos:

- '–opt tight' — criterio más estricto que el  predeterminado; necesario cuando la geometría sirve de entrada  a cálculos DFT.
- '–chrg 0' — carga total (0 para la cafeína neutra).
- '–uhf 0' — electrones sin aparear (0 para capa cerrada).
- '–gfn 2' — selecciona GFN2-xTB explícitamente para  reproducibilidad.

### 5. Extracción de parámetros geométricos

In [ ]:
import numpy as np

def leer_xyz(archivo):
    with open(archivo) as f:
        lineas = f.readlines()
    n   = int(lineas[0])
    atomos, coords = [], []
    for linea in lineas[2:2+n]:
        partes = linea.split()
        atomos.append(partes[0])
        coords.append([float(x) for x in partes[1:4]])
    return atomos, np.array(coords)

def distancia(coords, i, j):
    return np.linalg.norm(coords[i] - coords[j])

atomos, coords = leer_xyz('xtbopt.xyz')

# Ajustar índices (base 0) según la numeración del XYZ
# d_CO  = distancia(coords, idx_C, idx_O)
# print(f'C=O : {d_CO:.4f} Å')

### 6. Archivos generados

- 'cafeina\_2D.png' — imagen del grafo 2D.
- 'cafeina\_FF.xyz' — geometría MMFF94.
- 'cafeina\_xtb.out' — log completo de xTB.
- 'xtbopt.xyz' $\to$ renombrar a 'cafeina\_GFN2.xyz'.
- 'charges' — cargas de Mulliken por átomo.

## Control de calidad y validación

### Criterios de convergencia

### 📋 Información de la práctica

| Parámetro | Valor |
|:--|:--|
| **Mensaje de convergencia** | {GEOMETRY OPTIMIZATION CONVERGED |
| **Desplazamiento RMS** | $< 4 10^{-4 |
| **Frecuencias imaginarias** | 0 (verificar opcionalmente con {xtb –hess |

### Diagnósticos estructurales

Verificar que la geometría optimizada satisfaga:

- **Planaridad del anillo:** RMSD de los átomos del anillo  respecto al plano de mínimos cuadrados $< 0.05$~.
- **Longitudes de enlace características:**  C=O carbonilo: $1.20 \pm 0.02$~;  C–N aromático: $1.33 \pm 0.03$~;  C–N metilo: $1.46 \pm 0.02$~.
- **Sin colisiones:** distancia mínima entre no enlazados  $> 2.0$~.
- **Hidrógenos en posición escalonada** respecto al enlace  C–N adyacente en los grupos metilo.

### Validación frente a la estructura cristalina

La estructura cristalina de la cafeína monohidrato fue determinada por
difracción de rayos X (CSD: CAFINE01) {cite}`caffeine_crystal`. Completar
la columna "Calculado" y calcular el error porcentual:

| C2=O3 | 1.239 | … | … |
|:--|:--|:--|:--|
| N1–C8 | 1.374 | … | … |
| N3–C4 | 1.327 | … | … |

## Expansión del espacio químico (el bosque)

### Estrategia de expansión

El bosque contiene 50 moléculas orgánicas en cinco clases:

- **Aromáticos carbocíclicos** (10): benceno, naftaleno,  antraceno, pireno y derivados sustituidos.
- **Heteroaromáticos** (10): piridina, imidazol, tiofeno,  indol, quinolina y análogos.
- **Moléculas flexibles** (10): alcanos lineales  C$_6$–C$_{12}$, aminoácidos, dipéptidos.
- **Sistemas con puentes de H intramolecular** (10):  salicilaldehído, ácido 2-aminobenzoico y análogos.
- **Casos difíciles** (10): espiro-compuestos, cubano,  biciclo[1.1.0]butano y sistemas con estereoquímica compleja.

### Script de automatización

In [ ]:
%%bash

#!/usr/bin/env python3
# p01_expansion.py
import os, time, subprocess, re
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors, rdmolfiles

df_in = pd.read_csv('p01_bosque_smiles.csv')
registros = []

for _, fila in df_in.iterrows():
    reg = {'id': fila['id'], 'smiles': fila['smiles'],
           'clase': fila['clase']}
    mol = Chem.MolFromSmiles(fila['smiles'])
    if mol is None:
        reg['embed_ok'] = False; registros.append(reg); continue

    reg['n_heavy']     = mol.GetNumHeavyAtoms()
    reg['n_rot_bonds'] = rdMolDescriptors.CalcNumRotatableBonds(mol)
    reg['mw']          = rdMolDescriptors.CalcExactMolWt(mol)

    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3(); params.randomSeed = 42
    ok = AllChem.EmbedMolecule(mol_h, params)
    if ok == -1:
        reg['embed_ok'] = False; registros.append(reg); continue
    reg['embed_ok'] = True

    ff = AllChem.MMFFGetMoleculeForceField(
             mol_h, AllChem.MMFFGetMoleculeProperties(mol_h))
    if ff is None:
        ff = AllChem.UFFGetMoleculeForceField(mol_h)
    ff.Minimize(maxIts=2000)
    reg['e_ff_kcalmol'] = ff.CalcEnergy()

    os.makedirs('bosque', exist_ok=True)
    xyz_in = f"bosque/{fila['id']}_FF.xyz"
    rdmolfiles.MolToXYZFile(mol_h, xyz_in)

    t0  = time.time()
    res = subprocess.run(
        ['xtb', f"{fila['id']}_FF.xyz", '--opt', 'tight',
         '--chrg', '0', '--uhf', '0', '--gfn', '2'],
        capture_output=True, text=True, cwd='bosque')
    reg['t_xtb_s'] = time.time() - t0

    match = re.search(r'TOTAL ENERGY\s+([-\d.]+)', res.stdout)
    reg['e_xtb_hartree'] = float(match.group(1)) if match else None
    registros.append(reg)

pd.DataFrame(registros).to_csv(
    'p01_bosque_resultados.csv', index=False)

### Dataset pre-calculado

El bosque completo se proporciona en 'p01\_bosque\_resultados.csv'.
El script anterior está disponible para comprender el pipeline y poder
extenderlo; no es necesario ejecutarlo durante la sesión de laboratorio.

```{admonition} 📝 Nota metodológica
:class: note

**Nota sobre reproducibilidad:** El dataset fue generado con
'randomSeed = 42' y 'xtb' 6.6.1. Cualquier versión
$\geq$ 6.6 debería reproducir los resultados dentro de $< 0.1$\,\
energías y $< 0.002$~ en geometrías.
```

## Construcción del dataset

### Variables del dataset

| 'smiles' | str | — | PubChem / manual |
|:--|:--|:--|:--|
| 'clase' | str | — | categoría estructural |
| 'n\_heavy' | int | — | RDKit |
| 'n\_rot\_bonds' | int | — | RDKit |
| 'mw' | float | Da | RDKit |
| 'embed\_ok' | bool | — | 1 = éxito |
| 'e\_ff\_kcalmol' | float | kcal/mol | MMFF94 |
| 'e\_xtb\_hartree' | float | Hartree | GFN2-xTB |
| 't\_xtb\_s' | float | s | tiempo CPU |
| 'rmsd\_ring\_A' | float |  | RMSD planaridad |

### Integrar la semilla al dataset

In [ ]:
import pandas as pd

df = pd.read_csv('p01_bosque_resultados.csv')

semilla = {
    'id'             : 'cafeina',
    'smiles'         : 'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
    'clase'          : 'heteroaromatico',
    'n_heavy'        : 24,
    'n_rot_bonds'    : 3,
    'mw'             : 194.0804,
    'embed_ok'       : True,
    'e_ff_kcalmol'   : ...,    # completar con tu valor
    'e_xtb_hartree'  : ...,    # completar con tu valor
    't_xtb_s'        : ...,    # completar con tu valor
    'rmsd_ring_A'    : ...,    # completar con tu valor
}

df_final = pd.concat([df, pd.DataFrame([semilla])],
                     ignore_index=True)
df_final.to_csv('p01_dataset_final.csv', index=False)
print(df_final.tail(3))

## Análisis de resultados

### Estadística descriptiva

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns

df = pd.read_csv('p01_dataset_final.csv')
print(df[['n_heavy','n_rot_bonds','e_xtb_hartree',
          't_xtb_s']].describe().round(4))
print(f"Tasa de éxito: {df['embed_ok'].mean()*100:.1f}

### Gráficos principales

1. **Dispersión: átomos pesados vs tiempo xTB.**  Pregunta: ¿el tiempo de optimización escala linealmente o  de forma superlineal con el tamaño de la molécula?  verbatim df_ok = df[df['embed_ok']] fig, ax = plt.subplots(figsize=(6, 4)) sc = ax.scatter(df_ok['n_heavy'], df_ok['t_xtb_s'],  c=df_ok['n_rot_bonds'], cmap='viridis',  edgecolors='k', linewidths=0.4) ax.set_xlabel('Número de átomos pesados') ax.set_ylabel('Tiempo GFN2-xTB / s') fig.colorbar(sc, label='Enlaces rotativos') plt.tight_layout() plt.savefig('p01_tiempo_vs_natom.pdf') verbatim
2. **Boxplot: energía xTB por clase estructural.**  Pregunta: ¿qué clase tiene la energía media más negativa por  átomo y cómo se relaciona eso con la estabilización electrónica?  verbatim fig, ax = plt.subplots(figsize=(7, 4)) clases = df_ok['clase'].unique() datos = [df_ok[df_ok['clase']==c]['e_xtb_hartree']  for c in clases] ax.boxplot(datos, labels=clases, vert=True) ax.set_ylabel('Energía GFN2-xTB / Hartree') plt.xticks(rotation=30, ha='right') plt.tight_layout() plt.savefig('p01_energia_por_clase.pdf') verbatim
3. **Mapa de calor de correlaciones.**  Pregunta: ¿qué variable del dataset predice mejor el tiempo  de optimización?  verbatim cols = ['n_heavy','n_rot_bonds','mw',  'e_xtb_hartree','t_xtb_s'] fig, ax = plt.subplots(figsize=(5, 4)) sns.heatmap(df_ok[cols].corr(), annot=True, fmt='.2f',  cmap='coolwarm', center=0, ax=ax) plt.tight_layout() plt.savefig('p01_correlaciones.pdf') verbatim
4. **Casos de fallo.**  Listar las moléculas donde 'embed\_ok = False'  e identificar el patrón estructural común.  verbatim fallos = df[~df['embed_ok']][['id','smiles','clase']] print(fallos.to_string()) verbatim

In [ ]:
df_ok = df[df['embed_ok']]
fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(df_ok['n_heavy'], df_ok['t_xtb_s'],
                c=df_ok['n_rot_bonds'], cmap='viridis',
                edgecolors='k', linewidths=0.4)
ax.set_xlabel('Número de átomos pesados')
ax.set_ylabel('Tiempo GFN2-xTB / s')
fig.colorbar(sc, label='Enlaces rotativos')
plt.tight_layout()
plt.savefig('p01_tiempo_vs_natom.pdf')

**Boxplot: energía xTB por clase estructural.**
 Pregunta: ¿qué clase tiene la energía media más negativa por
 átomo y cómo se relaciona eso con la estabilización electrónica?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
clases = df_ok['clase'].unique()
datos  = [df_ok[df_ok['clase']==c]['e_xtb_hartree']
          for c in clases]
ax.boxplot(datos, labels=clases, vert=True)
ax.set_ylabel('Energía GFN2-xTB / Hartree')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('p01_energia_por_clase.pdf')

**Mapa de calor de correlaciones.**
 Pregunta: ¿qué variable del dataset predice mejor el tiempo
 de optimización?

In [ ]:
cols = ['n_heavy','n_rot_bonds','mw',
        'e_xtb_hartree','t_xtb_s']
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(df_ok[cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax)
plt.tight_layout()
plt.savefig('p01_correlaciones.pdf')

**Casos de fallo.**
 Listar las moléculas donde 'embed\_ok = False'
 e identificar el patrón estructural común.

In [ ]:
fallos = df[~df['embed_ok']][['id','smiles','clase']]
print(fallos.to_string())

enumerate

## Interpretación química

La comparación de la geometría semilla con el cristal de cafeína permite
evaluar directamente la calidad del protocolo ETKDG + MMFF94 + GFN2-xTB.
Los valores calculados de C=O y C–N deberían estar dentro del 1–2\,\
de los valores cristalográficos, confirmando que GFN2-xTB es adecuado
como punto de partida para DFT. Si la hipótesis sobre los metilos es
correcta, el RMSD debería ser mayor para los carbonos $sp^3$ que para
los átomos del anillo aromático.

En el bosque, las moléculas de la clase "casos difíciles" deberían
mostrar la mayor tasa de fallo en la incrustación. Esto ocurre porque
ETKDG fue parametrizado con distribuciones extraídas de la CSD, donde
las geometrías muy tensionadas están estadísticamente subrepresentadas.
El algoritmo no "conoce" el cubano porque casi no hay cubanos en la
base de datos de entrenamiento.

El análisis de correlaciones debería revelar que el tiempo de optimización
xTB escala aproximadamente como $O(N^{2.5\text{--}3})$ con el número de
átomos, lo que tiene implicaciones directas para el diseño de pipelines
masivos. Para conjuntos de $10^3$–$10^5$ moléculas (Bloque~17), GFN2-xTB
sigue siendo manejable; para conjuntos mayores conviene usar GFN-FF como
filtro previo.

El boxplot por clase debería mostrar que las moléculas aromáticas tienen
energías absolutas más negativas por átomo que las alifáticas de tamaño
comparable, reflejando la estabilización por deslocalización electrónica.
Esta tendencia, observada aquí de forma semiempírica, reaparecerá de
manera más cuantitativa cuando se calculen energías de estructura
electrónica en el Bloque~2.

```{admonition} 📝 Nota metodológica
:class: note

**Conexión con prácticas posteriores:** La geometría
'cafeina\_GFN2.xyz' es el input directo de la Práctica~4
(optimización B3LYP/6-31G(d)). Guárdala. El bosque de esta práctica
es la base del análisis de espacio conformacional de la Práctica~3.
```

## Entregables

### Datos

- ****D1.**** 'cafeina\_GFN2.xyz' — geometría semilla  optimizada.
- ****D2.**** 'p01\_dataset\_final.csv' — bosque con  la fila de la semilla añadida y la columna 'rmsd\_ring\_A'  completada para las 10 moléculas aromáticas.
- ****D3.**** 'cafeina\_xtb.out' — log de xTB sin  modificar.

### Figuras

- ****F1.**** 'p01\_tiempo\_vs\_natom.pdf' —  dispersión tiempo vs átomos, coloreada por enlaces rotativos.
- ****F2.**** 'p01\_energia\_por\_clase.pdf' —  boxplot de energía GFN2-xTB por clase.
- ****F3.**** 'p01\_correlaciones.pdf' —  mapa de calor de correlaciones.
- ****F4.**** (Opcional) Captura de la estructura 3D en  Avogadro2 con etiquetas de longitudes de enlace C=O y C–N.

### Texto

- ****T1.**** Reporte ($\leq 2$ páginas): tabla de validación  geométrica vs cristal, F1 y F2 comentadas, interpretación  de los patrones del bosque.
- ****T2.**** Respuestas a las preguntas de discusión  ($\leq 150$ palabras por pregunta).

## Preguntas de discusión

1. **(Comprensión)** El SMILES de la cafeína es  'Cn1cnc2c1c(=O)n(c(=O)n2C)C'. Escribe el SMILES  de la teobromina (7-metilxantina) y la teofilina  (1,3-dimetilxantina). ¿Qué diferencia estructural codifica  cada SMILES respecto al de la cafeína?
2. **(Comprensión)** ¿Qué significan '–chrg 0'  y '–uhf 0' en el comando xTB? ¿Qué ocurriría  química y numéricamente si ejecutaras el cálculo con  '–uhf 2' para la cafeína?
3. **(Análisis)** Examina el log de xTB e identifica:  (a) el número de ciclos de optimización, (b) la energía en  el primer y en el último ciclo, y (c) el cambio geométrico  entre el inicio (MMFF94) y el final (GFN2-xTB). ¿Fue útil  la pre-optimización con campo de fuerza?
4. **(Análisis)** Del bosque, identifica las tres moléculas  con mayor RMSD de planaridad de anillo y las tres con menor.  ¿Qué factor estructural parece determinante en cada caso?
5. **(Metodológico)** ¿Por qué es importante fijar  'randomSeed = 42'? ¿Qué consecuencia tendría para  la reproducibilidad del bosque si cada estudiante usara  una semilla diferente?
6. **(Metodológico)** GFN2-xTB es un método semiempírico.  ¿En qué práctica del Bloque~2 aprenderás la diferencia entre  métodos semiempíricos y de primeros principios? ¿Qué  limitaciones específicas de GFN2-xTB esperas al describir  la electrónica de la cafeína?

## Extensiones del ejercicio

- **Familia de xantinas:** repetir el pipeline para  teobromina, teofilina, paraxantina y adenina. Construir un  dataset de los 20 derivados más simples de la xantina y  analizar tendencias geométricas en la familia.
- **Recuperar fallos:** para las moléculas del bosque  donde 'embed\_ok = False', intentar recuperar la  incrustación usando 'AllChem.EmbedMultipleConfs'  con 100 conformaciones. ¿Cuántos se recuperan y a qué costo?
- **Fragmento de ChEMBL:** descargar 1000 SMILES con  peso molecular $< 300$~Da de ChEMBL y ejecutar el pipeline  sobre ese conjunto. Analizar tasa de éxito y distribución  de tiempos. Conecta con la Práctica~47 (Bloque~17).
- **GFN1 vs GFN2:** repetir la optimización semilla con  '–gfn 1'. Comparar geometrías y tiempos.  ¿Justifica GFN2 su mayor costo?
- **Puerta de entrada a DFT:** usar  'cafeina\_GFN2.xyz' como input de la Práctica~4 y  cuantificar cuánto cambia la geometría al pasar de xTB a  B3LYP/6-31G(d).

## 📚 Referencias

- **[rdkit]** Landrum, G. *RDKit: Open-Source Cheminformatics*.
Disponible en: [https://www.rdkit.org](https://www.rdkit.org). Acceso: 2026.
- **[etkdg]** Riniker, S.; Landrum, G.~A. *Better Informed Distance
Geometry: Using What We Know To Improve Conformation Generation*.
*J. Chem. Inf. Model.* **55**(12), 2562–2574, 2015.
DOI: [DOI:10.1021/acs.jcim.5b00654](https://doi.org/10.1021/acs.jcim.5b00654).
- **[grimme_xtb]** Bannwarth, C.; Ehlert, S.; Grimme, S. *GFN2-xTB — An
Accurate and Broadly Parametrized Self-Consistent Tight-Binding
Quantum Chemical Method with Multipole Electrostatics and
Density-Dependent Dispersion Contributions*.
*J. Chem. Theory Comput.* **15**(3), 1652–1671, 2019.
DOI: [DOI:10.1021/acs.jctc.8b01176](https://doi.org/10.1021/acs.jctc.8b01176).
- **[caffeine_crystal]** Sutor, D.~J. *The crystal structure of caffeine*.
*Acta Crystallogr.* **11**(7), 453–458, 1958.
DOI: [DOI:10.1107/S0365110X58001286](https://doi.org/10.1107/S0365110X58001286).
- **[openbabel]** O'Boyle, N.~M. et al. *Open Babel: An open chemical
toolbox*. *J. Cheminform.* **3**(1), 33, 2011.
DOI: [DOI:10.1186/1758-2946-3-33](https://doi.org/10.1186/1758-2946-3-33).
- **[lewars]** Lewars, E.~G. *Computational Chemistry: Introduction to
the Theory and Applications of Molecular and Quantum Mechanics*.
Springer, 2024. DOI: [DOI:10.1007/978-3-031-51443-2](https://doi.org/10.1007/978-3-031-51443-2).

```{bibliography}
:filter: cited
```
